In [1]:
from utils import *
from configs import *
from copy import deepcopy
from evaluate import *

from fit_pMAE import *
from fit_ReMasker import *

import random

c:\Users\Bhavya Prajapati\miniconda3\envs\PMAE\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
set_all_seeds(42)

In [3]:
args = get_args_parser().parse_args([])

#### data

In [4]:
# CONFIGS
basedir = './'
dset = 'diabetes'

# key 1: pattern (full / quasi)
missing_pattern = 'full'  # quasi

# key 2: seed (0~9)
seed = 2

# LOAD
proc_new = torch.load(f'{basedir}/amputation/{dset}/new_proc.pkl')

X_num = proc_new['X_num']
X_cat = proc_new['X_cat']
d_numerical = proc_new['d_numerical']

# full, quasi
res_new = torch.load(f'{basedir}/amputation/{dset}/amputed.pkl')

# Now res_new exists, so these are valid
print(res_new.keys())
print(res_new['quasi'].keys())
print(res_new['full'].keys())

# Select pattern and seed
res_new_ = res_new[missing_pattern][seed]

X_init_new = res_new_['X_init'].float()
mask_new = res_new_['mask'] == 1

X_incomp_new = res_new_['X_incomp'].float()

n, d = X_init_new.shape
d_numerical = proc_new['d_numerical']

miss_cols_new = [
    i for i in range(d)
    if i not in res_new_['full_cols']
]

dict_keys(['d', 'd_numerical', 'quasi', 'full'])
dict_keys([9999, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
dict_keys([9999, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [5]:
if (n < 1000):
    batch_size = 128
elif (n < 2500):
    batch_size = 256
elif (n <5000):
    batch_size = 512
elif (n <10000):
    batch_size = 1024
elif (n <20000):
    batch_size = 2048
else:
    batch_size = 4096

#### model

In [6]:
print(args)

Namespace(dataset='california', batch_size=64, max_epochs=600, accum_iter=1, mask_ratio=0.5, embed_dim=32, depth=6, decoder_depth=4, num_heads=4, mlp_ratio=4.0, encode_func='linear', norm_field_loss=False, weight_decay=0.05, lr=None, blr=0.001, min_lr=1e-05, warmup_epochs=40, path='/data/', exp_name='test', device='cuda', seed=666, overwrite=True, pin_mem=True)


In [7]:
model_pmae = ProportionalMasker(args)
model_pmae.batch_size = batch_size
model_pmae.device = 'cpu'
model_pmae.max_epochs = 300

model_remasker = ReMasker(args)
model_remasker.batch_size = batch_size
model_remasker.device = 'cpu'
model_remasker.max_epochs = 300

In [8]:
# pMAE

model_pmae.old_loss = False
model_pmae.block_mlp = 0 # Mixer -0 / Transformer - None
model_pmae.new_imp = True

X_imputed1 = model_pmae.fit(pd.DataFrame(X_incomp_new))
                                                                                         

0.5
missing col index: [0 1 2 4 5 6 7 8 9]
p_obs mean:  tensor([0.8528, 0.4442, 0.5381, 1.0000, 0.9137, 0.8223, 0.5761, 0.4416, 0.2716,
        0.9365])


c:\Users\Bhavya Prajapati\miniconda3\envs\PMAE\lib\site-packages\torch\cuda\amp\grad_scaler.py:126: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
c:\Users\Bhavya Prajapati\miniconda3\envs\PMAE\lib\site-packages\torch\amp\autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Loss at epoch 0: 0.5423850414215214
1 epoch took: 0.4891982078552246 sec
Loss at epoch 20: 0.3144042468733732
Loss at epoch 40: 0.21575663995277422
Loss at epoch 60: 0.20270037283235012
Loss at epoch 80: 0.1920206029515062
Loss at epoch 100: 0.17803689995885663
Loss at epoch 120: 0.1637989862757723
Loss at epoch 140: 0.1699227450885639
Loss at epoch 160: 0.15501535007489725
Loss at epoch 180: 0.14537838085252947
Loss at epoch 200: 0.1538003419751576
Loss at epoch 220: 0.15120581207763453
Loss at epoch 240: 0.1355846497711668
Loss at epoch 260: 0.1544184040165638
Loss at epoch 280: 0.16527673998864403


In [9]:
# ReMasker

model_remasker.new_imp = False
model_remasker.device = 'cpu'
X_imputed2 = model_remasker.fit(pd.DataFrame(X_incomp_new))
                                                                                         

0.5
missing col index: [0 1 2 4 5 6 7 8 9]
p_obs mean:  tensor([0.8528, 0.4442, 0.5381, 1.0000, 0.9137, 0.8223, 0.5761, 0.4416, 0.2716,
        0.9365])
Loss at epoch 0: 0.5490700673779778
1 epoch took: 0.23250460624694824 sec
Loss at epoch 20: 0.2644284696087437
Loss at epoch 40: 0.25647676395600194
Loss at epoch 60: 0.21424414670233607
Loss at epoch 80: 0.1977745869501606
Loss at epoch 100: 0.22617635417778767
Loss at epoch 120: 0.1913922192822234
Loss at epoch 140: 0.20155747965541357
Loss at epoch 160: 0.18467600889167898
Loss at epoch 180: 0.1954018817467041
Loss at epoch 200: 0.19526429415902
Loss is nan,
Loss at epoch 220: 0.18764872042147318
Loss at epoch 240: 0.18524479871290406
Loss at epoch 260: 0.18630836105092205
Loss at epoch 280: 0.16938155073060407


#### evaluation
- d: number of columns (total)
- d_numerical: number of columns (numerical only)
- X_init_new: ground truth
- X_imputed: imputed value
- mask_new: missing mask (1: miss)
- miss_cols: target columns that are missing
- cat_exists: whether there are categorical variables

In [10]:
# pMAE

X_imputed = X_imputed1

score_dict = evaluator(X_init_new, X_imputed, mask_new, d, d_numerical, 
                       miss_cols = miss_cols_new, 
                       cat_exists = (d - d_numerical > 0) )

print(score_dict)

{'imp_acc': 0.27620255947113037, 'R2(numerical)': 0.20572787523269653, 'Acc(categorical)': 0.8399999737739563, 'WD': 0.15126990275118504, 'RMSE': 0.20370280308817906, 'RMSE(num)': 0.19757407848866035, 'RMSE(cat)': 0.4039577901337178}


In [11]:
# ReMasker

X_imputed = X_imputed2

score_dict = evaluator(X_init_new, X_imputed, mask_new, d, d_numerical, 
                       miss_cols = miss_cols_new, 
                       cat_exists = (d - d_numerical > 0) )
print(score_dict)

{'imp_acc': 0.12735235691070557, 'R2(numerical)': 0.038271404802799225, 'Acc(categorical)': 0.8399999737739563, 'WD': 0.14780578462195182, 'RMSE': 0.20403931037416154, 'RMSE(num)': 0.19728397714623744, 'RMSE(cat)': 0.4192565132049186}
